# Calcium Imaging Pipeline v2
**Excel-driven**: `Application_period.xlsx` is the single source of truth for which files to load, which to use for segmentation, and what the experimental conditions are.

- Set `EXPERIMENT_FILTER` to a sheet name (e.g. `'Min6 WT Exp1'`) to run one experiment.
- `None` = run all experiments in the Excel.
- Outputs saved to `{BASE_DIR}/{exp_name}/pipeline_output/`.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
METADATA_XLSX     = r'C:\Users\DColameo\Documents\dev\pyprojects\mps_ca\Application_period.xlsx'
BASE_DIR          = r'Z:\ephacoffice\DColameo\Ca_Anand_AllData'

# Set to an Excel sheet name (e.g. 'Min6 WT Exp1') to process one experiment.
# None = process all experiments listed in the Excel.
EXPERIMENT_FILTER = None

# Acquisition
SOURCE_FPS      = 4
TARGET_FPS      = 1
SPATIAL_BINNING = 2

# Segmentation
GAUSS_SIGMA        = 2.0
MIN_PEAK_DISTANCE  = 8
MIN_CELL_AREA      = 10
MAX_CELL_AREA      = 1000
LOW_SIGNAL_THRESH  = 0.5
SEG_SCORE_WEIGHTS  = (1/3, 1/3, 1/3)  # (n_rois, total_area, mean_activity_per_roi)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import re, json, csv
from pathlib import Path

import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage import filters, morphology, measure, segmentation, feature
from skimage.color import label2rgb

%matplotlib inline
plt.rcParams['figure.dpi'] = 110


def _num_key(p):
    return tuple(int(n) for n in re.findall(r'\d+', p.name))


def parse_excel_metadata(xlsx_path):
    import openpyxl, re as _re

    def _parse_dur(v):
        if v is None: return None
        try: return int(str(v).replace('s', '').strip())
        except: return None

    def _infer_sf(fname):
        m = _re.match(r'^(\d+_\d+)_', fname)
        return m.group(1) if m else None

    wb = openpyxl.load_workbook(xlsx_path)
    result = []
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        rows = list(ws.iter_rows(values_only=True))[1:]
        exp_dir = cur_app = cur_phase = cur_dur = None
        files = []
        for row in rows:
            main_folder, subfolder, filename, application, phase, time_val, _, frames, seg = row
            if main_folder: exp_dir = str(main_folder)
            if application: cur_app = str(application)
            if phase: cur_phase = str(phase)
            if time_val: cur_dur = _parse_dur(time_val)
            if filename:
                fname = str(filename)
                if not fname.endswith('.tif'): fname += '.tif'
                inferred = _infer_sf(str(filename))
                excel_sf  = str(subfolder) if subfolder else None
                files.append({
                    'subfolder':   inferred if inferred else excel_sf,
                    'filename':    fname,
                    'application': cur_app,
                    'phase':       cur_phase,
                    'duration_s':  cur_dur,
                    'use_for_seg': bool(seg),
                })
        if exp_dir:
            result.append({'sheet': sheet, 'exp_dir': exp_dir, 'files': files})
    print(f'Parsed {len(result)} experiments from Excel')
    return result


def compute_dff(F, fps, baseline_window_s=60, pct=8):
    win = max(1, int(baseline_window_s * fps))
    n, T = F.shape
    F0 = np.zeros_like(F)
    for i in range(n):
        for t in range(T):
            lo, hi = max(0, t - win//2), min(T, t + win//2 + 1)
            F0[i, t] = np.percentile(F[i, lo:hi], pct)
    return (F - F0) / (F0 + 1e-6), F0


print('Helpers loaded')

In [ ]:
def run_experiment_pipeline(
    exp_dir,
    excel_meta,
    source_fps=SOURCE_FPS,
    target_fps=TARGET_FPS,
    spatial_binning=SPATIAL_BINNING,
    gauss_sigma=GAUSS_SIGMA,
    min_peak_distance=MIN_PEAK_DISTANCE,
    min_cell_area=MIN_CELL_AREA,
    max_cell_area=MAX_CELL_AREA,
    low_signal_thresh=LOW_SIGNAL_THRESH,
    seg_score_weights=SEG_SCORE_WEIGHTS,
):
    exp_dir   = Path(exp_dir)
    out_dir   = exp_dir / 'pipeline_output'
    plots_dir = out_dir / 'plots'
    out_dir.mkdir(exist_ok=True)
    plots_dir.mkdir(exist_ok=True)

    def _log(msg): print(f'  [{exp_dir.name}] {msg}')
    def _savefig(fig, name):
        fig.savefig(str(plots_dir / name), dpi=110, bbox_inches='tight')
        plt.close(fig)

    ratio = source_fps // target_fps

    # ---- 1. File list from Excel ----------------------------------------
    file_entries = []  # (path, application, phase, use_for_seg)
    for fi in excel_meta['files']:
        p = exp_dir / fi['subfolder'] / fi['filename']
        if p.exists():
            file_entries.append((p, fi['application'], fi['phase'], fi['use_for_seg']))
        else:
            _log(f'  MISSING: {fi["subfolder"]}/{fi["filename"]}')
    if not file_entries:
        _log('No files found on disk — skipping'); return {'exp': exp_dir.name, 'n_cells': 0, 'error': 'no_files'}
    n_seg = sum(1 for *_, s in file_entries if s)
    _log(f'{len(file_entries)} files ({n_seg} for segmentation)')

    # ---- 2. Load + downsample + spatial bin ----------------------------
    def _proc(arr):
        a = arr.astype(np.float32)
        T, H, W = a.shape
        Tt = (T // ratio) * ratio
        a  = a[:Tt].reshape(Tt // ratio, ratio, H, W).mean(axis=1)
        if spatial_binning > 1:
            b = spatial_binning
            T2, H2, W2 = a.shape
            Hb, Wb = H2 // b, W2 // b
            a = a[:, :Hb*b, :Wb*b].reshape(T2, Hb, b, Wb, b).mean(axis=(2, 4))
        return a

    chunks = []
    frame_apps   = []
    frame_phases = []
    seg_mask     = []
    for p, app, phase, use_seg in file_entries:
        raw = tifffile.imread(str(p))
        if raw.ndim == 2: raw = raw[np.newaxis]
        c = _proc(raw); del raw
        n_f = c.shape[0]
        frame_apps.extend([app]   * n_f)
        frame_phases.extend([phase] * n_f)
        seg_mask.extend([use_seg] * n_f)
        chunks.append(c)
        _log(f'  {p.parent.name}/{p.name} -> {c.shape}  seg={use_seg}')
    mov = np.concatenate(chunks, axis=0); del chunks
    _log(f'Movie: {mov.shape}  {mov.nbytes/1e9:.2f} GB')

    # ---- 3. Bad frame removal ------------------------------------------
    fmeans = mov.mean(axis=(1, 2))
    med    = np.median(fmeans)
    bf_thr = med * low_signal_thresh
    bad    = np.where(fmeans < bf_thr)[0]
    good   = np.where(fmeans >= bf_thr)[0]
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(fmeans, lw=0.8, color='steelblue', label='Frame mean')
    ax.axhline(bf_thr, color='red', lw=1.2, ls='--', label=f'Threshold ({low_signal_thresh:.0%} x median)')
    if len(bad): ax.scatter(bad, fmeans[bad], color='red', s=20, zorder=5, label=f'Bad frames n={len(bad)}')
    ax.set_xlabel('Frame'); ax.set_ylabel('Mean intensity'); ax.legend(fontsize=8)
    ax.set_title(f'{exp_dir.name} — frame means'); plt.tight_layout()
    _savefig(fig, '01_frame_means.png')
    if len(bad):
        good_set = set(good.tolist())
        frame_apps   = [frame_apps[i]   for i in range(len(frame_apps))   if i in good_set]
        frame_phases = [frame_phases[i] for i in range(len(frame_phases)) if i in good_set]
        seg_mask     = [seg_mask[i]     for i in range(len(seg_mask))     if i in good_set]
        mov = mov[good]
        _log(f'Dropped {len(bad)} bad frames -> {mov.shape[0]} remaining')

    # ---- 4. Projections (seg files only) --------------------------------
    seg_idx  = np.array([i for i, s in enumerate(seg_mask) if s], dtype=int)
    proj_mov = mov[seg_idx] if 0 < len(seg_idx) < len(mov) else mov
    sfx      = ' (seg files)' if len(proj_mov) < len(mov) else ''
    _log(f'Projections: {len(proj_mov)}/{len(mov)} frames{sfx}')
    mean_img = proj_mov.mean(axis=0).astype(np.float32)
    max_img  = proj_mov.max(axis=0).astype(np.float32)
    std_img  = proj_mov.std(axis=0).astype(np.float32)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, img, title, cmap in zip(axes,
        [mean_img, max_img, std_img], [f'Mean{sfx}', f'Max{sfx}', f'Std{sfx}'],
        ['gray', 'gray', 'inferno']):
        p1, p99 = np.percentile(img, [1, 99])
        ax.imshow(img, cmap=cmap, vmin=p1, vmax=p99); ax.set_title(title); ax.axis('off')
    plt.suptitle(exp_dir.name); plt.tight_layout()
    _savefig(fig, '02_projections.png')

    # ---- 5. Segmentation base images ------------------------------------
    def _norm(img): return (img - img.min()) / (img.max() - img.min() + 1e-9)
    mn = _norm(mean_img); sn = _norm(std_img); xn = _norm(max_img)
    base_opts = {'mean': mn, 'std': sn, 'combined': mn*sn, 'max': xn, 'max_std': xn*sn}
    def _build(base):
        s = filters.gaussian(base, sigma=gauss_sigma); return _norm(s)
    def _binary(seg, th):
        b = morphology.remove_small_objects(seg > th, min_size=min_cell_area)
        return ndimage.binary_fill_holes(b)
    def _wseg(seg, binary):
        lm = feature.peak_local_max(seg*binary.astype(float), min_distance=min_peak_distance, labels=binary)
        mk = np.zeros_like(binary, dtype=int)
        if len(lm): mk[tuple(lm.T)] = 1
        mk = measure.label(mk)
        return segmentation.watershed(-seg, mk, mask=binary)
    def _lroi(labels):
        roi = np.zeros_like(labels)
        valid = [p.label for p in measure.regionprops(labels) if min_cell_area <= p.area <= max_cell_area]
        for new_id, old in enumerate(valid, 1): roi[labels == old] = new_id
        return roi
    def _seg(seg, binary): return _lroi(_wseg(seg, binary))
    mmap = {'otsu': filters.threshold_otsu, 'yen': filters.threshold_yen,
            'li': filters.threshold_li, 'triangle': filters.threshold_triangle,
            'isodata': filters.threshold_isodata}
    base_names = list(base_opts.keys()); method_names = list(mmap.keys())
    segs = {name: _build(base) for name, base in base_opts.items()}

    # ---- 6. 5x5 sweep with composite scoring ----------------------------
    # composite = norm(n_rois) + norm(total_area) + norm(mean_std_per_roi)
    nb_ = len(base_names); nm_ = len(method_names)
    roi_counts      = np.zeros((nb_, nm_), dtype=int)
    area_totals     = np.zeros((nb_, nm_), dtype=float)
    activity_scores = np.zeros((nb_, nm_), dtype=float)
    roi_store = {}
    for bi, bname in enumerate(base_names):
        for mi, mname in enumerate(method_names):
            seg_i = segs[bname]
            th    = mmap[mname](seg_i)
            roi   = _seg(seg_i, _binary(seg_i, th))
            n     = int(roi.max())
            roi_counts[bi, mi] = n
            if n > 0:
                area_totals[bi, mi] = float(np.sum(roi > 0))
                per_roi = [std_img[roi == lbl].mean() for lbl in range(1, n + 1)]
                activity_scores[bi, mi] = float(np.mean(per_roi))
            roi_store[(bname, mname)] = (roi, th)

    def _n01(a):
        r = a.max() - a.min(); return (a - a.min()) / (r + 1e-9)
    w_n, w_a, w_act = seg_score_weights
    composite = w_n*_n01(roi_counts.astype(float)) + w_a*_n01(area_totals) + w_act*_n01(activity_scores)
    best_bi, best_mi   = np.unravel_index(composite.argmax(), composite.shape)
    best_roi, best_thr = roi_store[(base_names[best_bi], method_names[best_mi])]
    best_seg_name      = base_names[best_bi]
    best_method_name   = method_names[best_mi]
    n_cells            = int(best_roi.max())
    _log(f'Best: {best_seg_name}+{best_method_name} -> {n_cells} ROIs  '
         f'(th={best_thr:.4f}, score={composite[best_bi,best_mi]:.3f})')

    # 5x5 grid
    fig, axes = plt.subplots(nb_, nm_, figsize=(nm_*3.8, nb_*3.8))
    for bi, bname in enumerate(base_names):
        for mi, mname in enumerate(method_names):
            ax = axes[bi, mi]
            roi, th = roi_store[(bname, mname)]
            ov = np.clip(label2rgb(roi, image=mn, bg_label=0, alpha=0.4), 0, 1)
            ax.imshow(ov)
            is_best = (bi == best_bi and mi == best_mi)
            ax.set_title(f"{'(best) ' if is_best else ''}n={roi_counts[bi,mi]}  sc={composite[bi,mi]:.2f}",
                         fontsize=8, color='gold' if is_best else 'white', fontweight='bold' if is_best else 'normal')
            if mi == 0: ax.set_ylabel(bname, fontsize=9)
            if bi == 0: ax.set_title(f'{mname}\n' + ax.get_title(), fontsize=8,
                                      color='gold' if is_best else 'white', fontweight='bold' if is_best else 'normal')
            for sp in ax.spines.values():
                sp.set_edgecolor('gold' if is_best else '#444'); sp.set_linewidth(3 if is_best else 0.5)
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    plt.suptitle(f'{exp_dir.name} — sweep  gold=best: {best_seg_name}+{best_method_name} n={n_cells}', y=1.01)
    plt.tight_layout(); _savefig(fig, '03_sweep_grid.png')

    # Score heatmaps
    fig, axes = plt.subplots(1, 4, figsize=(22, 4))
    for ax, data, title in zip(axes,
            [roi_counts, area_totals, activity_scores, composite],
            ['N ROIs', 'Total area (px)', 'Mean act/ROI', 'Composite score']):
        im = ax.imshow(data, cmap='viridis', aspect='auto')
        ax.set_xticks(range(nm_)); ax.set_xticklabels(method_names, rotation=30, ha='right')
        ax.set_yticks(range(nb_)); ax.set_yticklabels(base_names)
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(title, fontsize=9)
        fmt = (lambda v: f'{v:.0f}') if title != 'Mean act/ROI' else (lambda v: f'{v:.2f}')
        for bi in range(nb_):
            for mi in range(nm_):
                ax.text(mi, bi, fmt(data[bi,mi]), ha='center', va='center', fontsize=7,
                        color='black' if data[bi,mi] > data.max()*0.6 else 'white')
        ax.plot(best_mi, best_bi, '*', color='gold', ms=14, zorder=5)
    plt.suptitle(exp_dir.name); plt.tight_layout()
    _savefig(fig, '04_score_heatmaps.png')

    # ---- 7. Final ROI overlay -------------------------------------------
    roi_mask  = best_roi
    rprops    = measure.regionprops(roi_mask)
    centroids = np.array([[rp.centroid[0], rp.centroid[1]] for rp in rprops])
    overlay   = np.clip(label2rgb(roi_mask, image=mn, bg_label=0, alpha=0.4), 0, 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    p1_, p99_ = np.percentile(mean_img, [1, 99])
    axes[0].imshow(mean_img, cmap='gray', vmin=p1_, vmax=p99_)
    axes[0].set_title('Mean (seg files)'); axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title(f'ROIs n={n_cells} | {best_seg_name}+{best_method_name} th={best_thr:.4f}')
    axes[1].axis('off')
    for rp in rprops:
        y, x = rp.centroid
        axes[1].text(x, y, str(rp.label), color='white', fontsize=5, ha='center', va='center')
        axes[1].plot(x, y, '+', color='yellow', ms=3, mew=0.6)
    plt.suptitle(exp_dir.name); plt.tight_layout()
    _savefig(fig, '05_final_rois.png')

    # ---- 8. Extract F traces (ALL files) --------------------------------
    T_final   = mov.shape[0]
    time_axis = np.arange(T_final) / target_fps
    F = np.zeros((n_cells, T_final), dtype=np.float32)
    for i, rp in enumerate(rprops):
        ys, xs = np.where(roi_mask == rp.label)
        F[i]   = mov[:, ys, xs].mean(axis=1)

    n_show = min(10, n_cells)
    if n_show > 0:
        fig, axes = plt.subplots(n_show, 1, figsize=(14, max(4, n_show*1.2)), sharex=True)
        if n_show == 1: axes = [axes]
        for i, ax in enumerate(axes):
            ax.plot(time_axis, F[i], lw=0.8); ax.set_ylabel(f'Cell {i+1}', fontsize=7)
        axes[-1].set_xlabel('Time (s)')
        plt.suptitle(f'{exp_dir.name} — Raw F (first {n_show} cells)')
        plt.tight_layout(); _savefig(fig, '06_raw_traces.png')

    # ---- 9. dF/F --------------------------------------------------------
    _log('Computing dF/F ...')
    dff, _ = compute_dff(F, target_fps)
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    vmax = np.percentile(np.abs(dff), 99)
    im = axes[0].imshow(dff, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                        extent=[0, time_axis[-1], n_cells+0.5, 0.5])
    axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Cell #'); axes[0].set_title('dF/F heatmap')
    plt.colorbar(im, ax=axes[0], label='dF/F')
    step = max(np.percentile(np.abs(dff), 95)*3, 0.1)
    for i in range(n_cells): axes[1].plot(time_axis, dff[i]+i*step, lw=0.7)
    axes[1].set_xlabel('Time (s)'); axes[1].set_title('dF/F traces')
    axes[1].set_yticks(np.arange(n_cells)*step)
    axes[1].set_yticklabels([f'Cell {i+1}' for i in range(n_cells)], fontsize=6)
    plt.suptitle(exp_dir.name); plt.tight_layout()
    _savefig(fig, '07_dff.png')

    # ---- 10. Save -------------------------------------------------------
    np.save(str(out_dir / 'roi_mask.npy'),  roi_mask)
    np.save(str(out_dir / 'centroids.npy'), centroids)
    np.save(str(out_dir / 'F_raw.npy'),     F)
    np.save(str(out_dir / 'dff.npy'),       dff)
    np.save(str(out_dir / 'time_axis.npy'), time_axis)
    tifffile.imwrite(str(out_dir / 'mean_image.tif'), mean_img)
    with open(str(out_dir / 'centroids.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['cell_id', 'y_pixel', 'x_pixel', 'area_px'])
        for rp in rprops:
            w.writerow([rp.label, round(rp.centroid[0],2), round(rp.centroid[1],2), rp.area])

    n_plots = len(list(plots_dir.glob('*.png')))
    _log(f'Done — {n_cells} cells, {T_final} frames, {n_plots} plots saved to {out_dir}')
    return {
        'exp'      : exp_dir.name,
        'sheet'    : excel_meta['sheet'],
        'n_cells'  : n_cells,
        'seg_image': best_seg_name,
        'method'   : best_method_name,
        'threshold': round(float(best_thr), 4),
        'score'    : round(float(composite[best_bi, best_mi]), 4),
        'n_frames' : T_final,
        'n_bad'    : int(len(bad)),
    }


print('run_experiment_pipeline() defined')

In [ ]:
import traceback
try: import pandas as pd; _HAS_PD = True
except ImportError: _HAS_PD = False

xl_meta   = parse_excel_metadata(METADATA_XLSX)
base_path = Path(BASE_DIR)

# Filter to one experiment if EXPERIMENT_FILTER is set
if EXPERIMENT_FILTER:
    xl_meta = [m for m in xl_meta
               if m['sheet'] == EXPERIMENT_FILTER or m['exp_dir'] == EXPERIMENT_FILTER]
    if not xl_meta:
        raise ValueError(f'EXPERIMENT_FILTER={EXPERIMENT_FILTER!r} not found. '
                         f'Available sheets: {[m["sheet"] for m in parse_excel_metadata(METADATA_XLSX)]}')

exp_entries = []
for m in xl_meta:
    d = base_path / m['exp_dir']
    if d.is_dir(): exp_entries.append((d, m))
    else: print(f'WARNING: not on disk: {m["exp_dir"]}')

print(f'Base: {base_path}')
print(f'Processing {len(exp_entries)} / {len(xl_meta)} experiments:')
for d, m in exp_entries:
    n_seg = sum(1 for f in m['files'] if f['use_for_seg'])
    print(f'  [{m["sheet"]:18s}]  {d.name}  ({len(m["files"])} files, {n_seg} for seg)')

results = []
for exp_dir, meta in exp_entries:
    print(f"\n{'='*60}\nSTART: {exp_dir.name}")
    try:
        r = run_experiment_pipeline(exp_dir, excel_meta=meta)
    except Exception as e:
        print(f'  ERROR: {e}'); traceback.print_exc()
        r = {'exp': exp_dir.name, 'n_cells': -1, 'error': str(e)}
    results.append(r)

print(f"\n{'='*60}\nBATCH COMPLETE")
if _HAS_PD:
    summary = pd.DataFrame(results)
    print(summary.to_string(index=False))
    summary
else:
    for r in results: print(r)